# Week 7 Live Coding
## Three numbers, one experiment

The vendor says +5.5 percentage points. The literature says +0.1 to +0.3. Both are computed from the same data. Today we'll see why they disagree and which one to trust.

Three things we will compute:
1. **The naive complier comparison** — the vendor's headline number (biased)
2. **The intent-to-treat effect (ITT)** — the honest number
3. **The LATE** — the effect on viewers, estimated honestly

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk07_itt_vs_late/data/ad_experiment.csv')
df.head()

In [ ]:
df.shape

In [ ]:
# What's in the data?
# assigned_treatment: 1 if randomly assigned to be served ads, 0 if control
# viewed_ad: 1 if the voter actually viewed at least one ad, 0 if not
# turned_out: 1 if the voter turned out to vote, 0 if not
print('Treatment group size:', df['assigned_treatment'].sum())
print('Control group size:  ', (df['assigned_treatment'] == 0).sum())

## Part 1 — Reproduce the vendor's headline number

The vendor compared turnout among voters who **viewed the ad** to turnout among the **control group**. Let's reproduce that.

In [ ]:
# Step 1: What fraction of the treatment group actually viewed the ad?
treatment_group = df[df['assigned_treatment'] == 1]
compliance_rate = treatment_group['viewed_ad'].mean()
print(f'Compliance rate: {compliance_rate:.1%}')
print(f'Viewers: {treatment_group["viewed_ad"].sum():,} out of {len(treatment_group):,}')

About one-third of the treatment group actually saw the ad. The other two-thirds were served ads but never viewed them (ad blockers, didn't visit the platform, etc.).

Now: the vendor's comparison.

In [ ]:
# The vendor's headline number: viewers vs. control
viewer_turnout = df[(df['assigned_treatment'] == 1) & (df['viewed_ad'] == 1)]['turned_out'].mean()
control_turnout = df[df['assigned_treatment'] == 0]['turned_out'].mean()
naive_gap = viewer_turnout - control_turnout

print(f'Turnout among ad viewers:  {100*viewer_turnout:.1f}%')
print(f'Turnout among control:     {100*control_turnout:.1f}%')
print(f'Vendor\'s headline:         {100*naive_gap:+.1f} percentage points')

**Stop.** That +5.5 point gap is the vendor's headline number. It's real — those viewers really did turn out 5.5 points more than the control group.

But think about **who** the viewers are. They're the 33% of the treatment group who were online enough, engaged enough, and on the right platforms to actually see the ad. What about the 67% who didn't view it?

In [ ]:
# What does the treatment group look like if we split by compliance?
non_viewer_turnout = df[(df['assigned_treatment'] == 1) & (df['viewed_ad'] == 0)]['turned_out'].mean()

print(f'Turnout among viewers:      {100*viewer_turnout:.1f}%')
print(f'Turnout among non-viewers:  {100*non_viewer_turnout:.1f}%')
print(f'Turnout among control:      {100*control_turnout:.1f}%')

**Read those three numbers carefully.** The viewers turn out at \~50%. The non-viewers turn out at \~42%. The control group turns out at \~45%.

The viewers were *already* more likely to vote — they're more politically engaged, more online, more reachable. The vendor's +5.5 point gap is mostly measuring this engagement difference, not the ad's effect.

**This is the Week 2 mistake in a new costume.** In Week 2, the vendor compared districts that chose to run the program to districts that didn't — and the choosers were already different. Here, the vendor compares voters who chose to view the ad to voters who didn't — and the viewers were already different.

The moment you condition on who complied, you've thrown away the randomization and you're back in Week 2.

## Part 2 — The intent-to-treat effect (ITT)

The randomization assigned voters to treatment or control. It did NOT determine who actually viewed the ad. The only comparison that respects the randomization is:

**Everyone assigned to treatment vs. everyone assigned to control.**

This is called the **intent-to-treat (ITT)** effect — the effect of being *assigned* to the treatment group, not the effect of *receiving* the treatment.

In [ ]:
# The ITT: compare EVERYONE assigned to treatment vs. EVERYONE in control.
# This is just a groupby on assigned_treatment — the same move from Weeks 3-5.
turnout_by_assignment = df.groupby('assigned_treatment')['turned_out'].mean()
print(turnout_by_assignment)

itt = turnout_by_assignment.loc[1] - turnout_by_assignment.loc[0]
print(f'\nITT: {100*itt:+.2f} percentage points')

The ITT is about **+0.1 percentage points**. That is the honest causal effect of being assigned to the ad campaign.

Why is it so small? Because only 33% of the treatment group actually viewed the ad. The effect of the ad (on the people who saw it) is "diluted" across the full treatment group, including the 67% who never saw it.

The ITT is real, it's honest, and it's tiny. The vendor's +5.5 is inflated by selection. The question is: can we recover the actual effect on viewers without falling into the vendor's trap?

## Part 3 — The LATE (effect on viewers, estimated honestly)

The **local average treatment effect (LATE)** estimates the effect on the people who actually viewed the ad — but using the randomization, not the biased comparison.

The intuition: the ITT is the ad's effect, diluted by noncompliance. If only 33% of the treatment group saw the ad, then the ITT captures 33% of the effect. To "undilute" it, divide by the compliance rate:

**LATE = ITT / compliance rate**

In [ ]:
# LATE = ITT / compliance rate
# The compliance rate tells us what fraction of the treatment group actually got treated.
# Dividing "undilutes" the ITT to recover the effect on the people who were exposed.

late = itt / compliance_rate

print(f'ITT:              {100*itt:+.2f} pp')
print(f'Compliance rate:  {compliance_rate:.1%}')
print(f'LATE:             {100*late:+.2f} pp')

The LATE is about **+0.3 percentage points** — consistent with the published literature (Aggarwal et al. found +0.1 to +0.3 pp). The ad has a real but very small effect on the people who view it.

Compare:

| Estimate | Value | Valid? |
|---|---|---|
| Vendor's headline (viewers vs. control) | +5.5 pp | **No** — viewers are self-selected |
| ITT (assigned treatment vs. control) | +0.1 pp | **Yes** — respects randomization |
| LATE (ITT / compliance) | +0.3 pp | **Yes** — recovers effect on viewers honestly |

The vendor's number is **18× too large**. The real effect is 0.3 points, not 5.5.

## Part 4 — What does this mean for the \$3M contract?

Let's compute the cost per additional vote under the honest numbers.

In [ ]:
# Cost per additional vote under the ITT
total_cost = 3_000_000
n_treated = df['assigned_treatment'].sum()
additional_votes_itt = itt * n_treated
cost_per_vote_itt = total_cost / additional_votes_itt

print(f'ITT: {100*itt:+.2f} pp on {n_treated:,} voters = {additional_votes_itt:.0f} additional votes')
print(f'Cost per additional vote (ITT): \${cost_per_vote_itt:,.0f}')

# Compare to tactics from earlier weeks
print(f'\nFor comparison:')
print(f'  W3 mailer (Voting Report Card): ~\$143 per vote')
print(f'  W5 canvassing:                  ~\$1,000 per vote')
print(f'  This digital ad campaign:       ~\${cost_per_vote_itt:,.0f} per vote')

That's the honest assessment for the client. The ad works — a little. But at \$60,000 per additional vote, it's extraordinarily expensive compared to the alternatives.

The vendor's report made the campaign look effective by comparing a self-selected subgroup (viewers) to the control group. The honest comparison (ITT or LATE) shows the effect is real but far too small to justify \$3M.

---

## What you've seen today

- **The naive complier comparison** compares people who chose to comply (viewers) to the control group. It's the Week 2 mistake applied to an experiment: conditioning on a post-randomization variable (who viewed) throws away the randomization.
- **The ITT** compares everyone assigned to treatment vs. everyone assigned to control. It respects the randomization. It's the honest number, but it's "diluted" by noncompliance.
- **The LATE = ITT / compliance rate** recovers the effect on compliers (viewers) without conditioning on who complied.
- All three numbers come from the same `groupby` tool you've used since Week 2. The difference is *what* you compute it on.

**Lying-with-data tag #6:** Reporting the complier comparison as a causal effect.

Next, open `problem_set.ipynb`.